In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/home/yuanshanwu/Documents/GitHub/QuantUS/engines/ceus/CLI-Demos
/home/yuanshanwu/Documents/GitHub/QuantUS/engines/ceus


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [2]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['mp4', 'nifti', 'avi']


In [3]:
scan_type = 'nifti'

# Takes the DICOM file as input for contrast enhanced ultrasound (CEUS) scans
CEUS_scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/UCSD-P05-V02-CE1_09.45.08_mf_sip_capture_50_2_1_0_CEUS.nii'
bmode_scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/UCSD-P05-V02-CE1_09.45.08_mf_sip_capture_50_2_1_0_BMODE.nii'
scan_loader_kwargs = {
}

In [4]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, CEUS_scan_path, **scan_loader_kwargs)
bmode_image_data = scan_loading_step(scan_type, bmode_scan_path, **scan_loader_kwargs)

In [5]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['load_bolus_mask', 'nifti']


In [6]:
seg_type = 'nifti'


seg_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/motion_VOI.nii.gz'
seg_loader_kwargs = {}

In [7]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, CEUS_scan_path, **seg_loader_kwargs)

# Load data

In [ ]:
import pickle
with open('/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/ax_5_sag_2.5_cor_2.5_with_mc_correlation_based.pkl', 'rb') as f:
    seg_data = pickle.load(f)

# CEUS Quantitative Temporal Curve Visualization

In [9]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))
print("Available analysis functions:", list(all_analysis_funcs.keys()))

Available analysis types: ['curves', 'curves_paramap']
Available analysis functions: ['tic', 'pyradiomics']


In [10]:
analysis_type = 'curves_paramap'
analysis_funcs = ['tic']

# Find all required kwargs for the analysis functions
analysis_funcs = analysis_funcs if len(analysis_funcs) else list(all_analysis_funcs[analysis_type].keys())
required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

Required kwargs for current analysis: ['cor_vox_len', 'ax_vox_ovrlp', 'ax_vox_len', 'sag_vox_ovrlp', 'sag_vox_len', 'cor_vox_ovrlp']


In [15]:
analysis_kwargs = {
    'ax_vox_ovrlp': 50,  # %
    'sag_vox_ovrlp': 50,  # %
    'cor_vox_ovrlp': 50,  # %
    'ax_vox_len': 10.0,  # mm
    'sag_vox_len': 5.0,  # mm
    'cor_vox_len': 5.0,  # mm
    'curves_output_path': '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/curve_map.csv', # don't export the curves we generate
}

In [16]:
from src.entrypoints import analysis_step

analysis_obj = analysis_step(analysis_type, image_data, seg_data, analysis_funcs, **analysis_kwargs)

Computing curves: 100%|██████████| 530/530 [44:36<00:00,  5.05s/it]


In [19]:
# Save the segmentation data with motion compensation applied
import pickle
from pathlib import Path

# Build output path: split CEUS_scan_path at 'HighQuality' and append pkl filename
base_path =  '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01' #CEUS_scan_path.split('HighQuality')[0]
output_pkl_path = str(Path(base_path)  / 'ax_5_sag_2.5_cor_2.5_with_mc_correlation_based.pkl')
print(f"Saving to: {output_pkl_path}")
# Save seg_data with motion compensation
with open(output_pkl_path, 'wb') as f:
    pickle.dump(analysis_obj, f)

Saving to: /home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/ax_5_sag_2.5_cor_2.5_with_mc_correlation_based.pkl


## Curve Quantification for each sub-kernels

In [ ]:
import pickle
with open('/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/ax_5_sag_2.5_cor_2.5_with_mc_correlation_based.pkl', 'rb') as f:
    analysis_obj = pickle.load(f)

In [9]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

Available quantification functions: dict_keys(['lognormal_fit_full', 'dte', 'wash_rates', 'first_order_select', 'first_order_full', 'cmus_firstorder', 'auc_no_fit', 'lognormal_fit_select'])


In [10]:
function_names = ['lognormal_fit_full'] # Empty list will use all functions
output_path = 'test_quants.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
}

In [11]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

## Visualize the parametric Map

In [12]:
from src.visualizations.options import get_visualization_types

types, funcs = get_visualization_types()
print("Available visualization types:", list(types.keys()))
print("Available visualization functions:", list(funcs.keys()))

Available visualization types: ['paramap']
Available visualization functions: ['paramap']


In [13]:
vis_type = 'paramap'
params = [] # Empty list will use all parameters 
vis_funcs = []
vis_kwargs = {
    'paramap_folder_path': 'paramaps_LOGNORMAL',
    'hide_all_visualizations': False,  # Set to True to hide all visualizations
}

In [14]:
from src.entrypoints import visualization_step

vis_obj = visualization_step(curve_quant, vis_type, params, vis_funcs, **vis_kwargs)

In [26]:
vis_obj.paramap_names

['AUC_full_TIC',
 'PE_full_TIC',
 'TP_full_TIC',
 'MTT_full_TIC',
 'T0_full_TIC',
 'Mu_full_TIC',
 'Sigma_full_TIC',
 'PE_Ix_full_TIC']

In [23]:
from src.map_showing import *
import os
os.environ["QT_API"] = "pyqt6"

# 3. Overlay T0 on CEUS image (with motion compensation!)
viewer = view_T0_with_CEUS(vis_obj, image_data, 
                          param_name='T0_full_TIC',
                          time_point=50)  # Frame 50
napari.run()

# Pixel-wise T0 map

In [8]:
seg_data.use_mc = False
seg_data.seg_mask.shape

(295, 218, 219)

In [9]:
from src.pixel_T0 import generate_t0_map_3d, get_t0_statistics_3d

t0_map = generate_t0_map_3d(
    image_data, seg_data,
    threshold=100, start_frame=0, end_frame=200,
    min_consecutive_frames=3
)
stats = get_t0_statistics_3d(t0_map, seg_data)

In [10]:
print("Non Motion Compensated T0 Map Statistics:")
print(stats)

Non Motion Compensated T0 Map Statistics:
{'mean_t0': 106.2957763671875, 'median_t0': 107.0, 'std_t0': 17.78708267211914, 'min_t0': 3.0, 'max_t0': 156.0, 'coverage': 98.38486040846918}


In [10]:
from src.map_showing import *
import os
os.environ["QT_API"] = "pyqt6"

# 3. Overlay T0 on CEUS image (with motion compensation!)
viewer = view_heatmap(t0_map, image_data, 
                      time_point=50, seg_mask=seg_data.seg_mask, title_name='T0 map manual VOI')  #50
napari.run()

T0 Map: contrast=[0, 142.0], mean(activated)=77.6, activated voxels=14888


In [11]:
from src.pixel_T0 import generate_t0_map_3d, get_t0_statistics_3d
seg_data.use_mc = True
t0_map_mc = generate_t0_map_3d(
    image_data, seg_data,
    threshold=100, start_frame=0, end_frame=200,
    min_consecutive_frames=3
)
stats_mc = get_t0_statistics_3d(t0_map_mc, seg_data)
print("Motion Compensated T0 Map Statistics:")
print(stats_mc)

Motion Compensated T0 Map Statistics:
{'mean_t0': 77.65586853027344, 'median_t0': 78.0, 'std_t0': 39.47584533691406, 'min_t0': 3.0, 'max_t0': 159.0, 'coverage': 17.966514577372205}


In [12]:
from src.map_showing import *
import os
os.environ["QT_API"] = "pyqt6"
viewer = view_heatmap(t0_map_mc, image_data, 
                      time_point=50, seg_mask=seg_data.seg_mask, title_name='T0 map medsam2 VOI')  #0
napari.run()

T0 Map: contrast=[0, 140.0], mean(activated)=71.6, activated voxels=13046
